# Qwen3-8B `down_proj` → TT-matrix 验证

这个 notebook 只读取 `model.layers.0.mlp.down_proj.weight`，验证：

1. 稠密矩阵与 tensorized matrix 可以精确往返；
2. `tt_svd_matrix()` 能生成 TT-matrix/MPO cores；
3. `TTMatrixLinear` 的直接收缩与重构矩阵乘法一致。

目标权重形状为 `[4096, 12288]`。

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/xls/workspace/projects/qwen3-tn-compression").resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

assert SRC_DIR.is_dir(), f"源码目录不存在：{SRC_DIR}"
print("项目目录：", PROJECT_ROOT)
print("源码目录：", SRC_DIR)

项目目录： /mnt/intern7/xls/projects/qwen3-tn-compression
源码目录： /mnt/intern7/xls/projects/qwen3-tn-compression/src


In [2]:
import time

import torch
from torch.nn import functional as F

from qwen3_tn import (
    TTMatrixLinear,
    TTMatrixSpec,
    detensorize_matrix,
    reconstruct_matrix,
    tensorize_matrix,
    tt_svd_matrix,
)
from qwen3_tn.experiment import load_indexed_weight

print("PyTorch：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available())

PyTorch： 2.5.1
CUDA 可用： True


## 1. 只读取 Qwen3-8B 第 0 层 `down_proj`

这里不加载整个 Qwen3 模型，只从 checkpoint 中读取目标权重。

In [3]:
MODEL_PATH = Path("/infini-data/Qwen3-8B")
TENSOR_NAME = "model.layers.0.mlp.down_proj.weight"

assert MODEL_PATH.is_dir(), f"模型目录不存在：{MODEL_PATH}"

weight = load_indexed_weight(str(MODEL_PATH), TENSOR_NAME)

print("权重名称：", TENSOR_NAME)
print("权重形状：", tuple(weight.shape))
print("权重 dtype：", weight.dtype)
print("权重 device：", weight.device)

assert tuple(weight.shape) == (4096, 12288)

权重名称： model.layers.0.mlp.down_proj.weight
权重形状： (4096, 12288)
权重 dtype： torch.bfloat16
权重 device： cpu


## 2. 定义 TT-matrix 规格

`out_modes` 的乘积必须等于 4096，`in_modes` 的乘积必须等于 12288。这里为三个内部 bond 分别设置 `64`、`512`、`192`。

In [4]:
spec = TTMatrixSpec(
    out_modes=(8, 8, 8, 8),
    in_modes=(8, 8, 8, 24),
    # ranks=(1, 64, 512, 192, 1),
    ranks=(1, 64, 4096, 192, 1),
)

assert spec.out_features == weight.shape[0]
assert spec.in_features == weight.shape[1]

print("TT-matrix ranks：", spec.ranks)
print("TT-matrix 参数量：", f"{spec.num_parameters:,}")
print("稠密矩阵参数量：", f"{spec.dense_num_parameters:,}")
print("理论压缩率：", f"{spec.compression_ratio:.2f}x")

TT-matrix ranks： (1, 64, 4096, 192, 1)
TT-matrix 参数量： 67,149,824
稠密矩阵参数量： 50,331,648
理论压缩率： 0.75x


## 3. 验证矩阵张量化

In [5]:
tensorized = tensorize_matrix(weight, spec)
round_trip = detensorize_matrix(tensorized, spec)

print("张量化形状：", tuple(tensorized.shape))
print("能否精确恢复：", torch.equal(round_trip, weight))

assert torch.equal(round_trip, weight)
del tensorized, round_trip

张量化形状： (64, 64, 64, 192)
能否精确恢复： True


## 4. 执行 TT-matrix 分解

In [6]:
if not torch.cuda.is_available():
    raise RuntimeError("真实 Qwen3 权重分解需要 CUDA GPU")

device = torch.device("cuda:0")
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
free_gib = free_bytes / 1024**3
total_gib = total_bytes / 1024**3

print("GPU：", torch.cuda.get_device_name(device))
print(f"空闲/总显存：{free_gib:.2f}/{total_gib:.2f} GiB")

required_free_gib = 60
if free_gib < required_free_gib:
    raise RuntimeError(
        f"空闲显存不足：需要至少 {required_free_gib} GiB，当前为 {free_gib:.2f} GiB"
    )

GPU： NVIDIA A100-SXM4-80GB
空闲/总显存：78.02/79.15 GiB


In [7]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

weight_fp32 = weight.float().to(device)

torch.cuda.reset_peak_memory_stats(device)
started = time.perf_counter()

cores = tt_svd_matrix(
    weight_fp32,
    spec,
    svd_driver="gesvd",
)

torch.cuda.synchronize(device)
elapsed = time.perf_counter() - started
peak_gib = torch.cuda.max_memory_allocated(device) / 1024**3

print(f"分解耗时：{elapsed:.2f} 秒")
print(f"PyTorch 峰值分配显存：{peak_gib:.2f} GiB")

分解耗时：6.94 秒
PyTorch 峰值分配显存：1.63 GiB


## 5. 检查 TT-matrix cores

In [8]:
for index, core in enumerate(cores):
    print(
        f"core {index}: shape={tuple(core.shape)}, "
        f"dtype={core.dtype}, device={core.device}"
    )

# expected_shapes = [
#     (1, 8, 8, 64),
#     (64, 8, 8, 512),
#     (512, 8, 8, 192),
#     (192, 8, 24, 1),
# ]

expected_shapes = [
    (1, 8, 8, 64),
    (64, 8, 8, 4096),
    (4096, 8, 8, 192),
    (192, 8, 24, 1),
]

assert [tuple(core.shape) for core in cores] == expected_shapes

core 0: shape=(1, 8, 8, 64), dtype=torch.float32, device=cuda:0
core 1: shape=(64, 8, 8, 4096), dtype=torch.float32, device=cuda:0
core 2: shape=(4096, 8, 8, 192), dtype=torch.float32, device=cuda:0
core 3: shape=(192, 8, 24, 1), dtype=torch.float32, device=cuda:0


## 6. 测量权重重构误差

In [9]:
reconstructed = reconstruct_matrix(cores, spec)

weight_relative_error = (
    torch.linalg.vector_norm(reconstructed - weight_fp32)
    / torch.linalg.vector_norm(weight_fp32)
)

print("权重相对 L2 误差：", weight_relative_error.item())
assert torch.isfinite(weight_relative_error)

权重相对 L2 误差： 4.174844889348606e-06


## 7. 验证 `TTMatrixLinear.forward()`

需要区分两种误差：

- `TTMatrixLinear` 与重构矩阵的误差：验证直接收缩实现；
- `TTMatrixLinear` 与原权重的误差：衡量压缩近似误差。

In [10]:
layer = TTMatrixLinear(
    spec,
    cores,
    token_chunk_size=8,
    trainable=False,
    preserve_input_dtype=False,
)

inputs = torch.randn(2, 4, spec.in_features, device=device, dtype=torch.float32)

with torch.inference_mode():
    tt_output = layer(inputs)
    reconstructed_output = F.linear(inputs, reconstructed)
    dense_output = F.linear(inputs, weight_fp32)

direct_relative_error = (
    torch.linalg.vector_norm(tt_output - reconstructed_output)
    / torch.linalg.vector_norm(reconstructed_output)
)
compression_relative_error = (
    torch.linalg.vector_norm(tt_output - dense_output)
    / torch.linalg.vector_norm(dense_output)
)

print("直接收缩 vs 重构矩阵，相对 L2 误差：", direct_relative_error.item())
print("TT-matrix vs 原始矩阵，相对 L2 误差：", compression_relative_error.item())

assert direct_relative_error.item() < 1e-4
assert torch.isfinite(compression_relative_error)

直接收缩 vs 重构矩阵，相对 L2 误差： 2.4067364847724093e-06
TT-matrix vs 原始矩阵，相对 L2 误差： 4.47819365945179e-06


## 8. 释放 GPU 内存

In [11]:
del layer, inputs, tt_output, reconstructed_output, dense_output
del reconstructed, cores, weight_fp32
torch.cuda.empty_cache()

print("GPU 临时张量已释放")

GPU 临时张量已释放
